In [5]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import ruptures as rpt
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ============================================================
# Pairwise GGS-like Robust Change Point Detection
# ------------------------------------------------------------
# Input  : ./tvpvar_input_scaled.csv
# Output : ./result_pair_robust/
#
# Method:
# - SOLVPN-X pairwise regime detection
# - min_size = 30, 60, 90, 120 robustness check
# - PELT-based candidate generation
# - Gaussian NLL + BIC-style criterion for model selection
# - No arbitrary target cp count heuristic
# ============================================================

# -----------------------------
# Config
# -----------------------------
BASE_DIR = Path("./")
RESULT_DIR = BASE_DIR / "result_pair_robust"
PLOT_DIR = RESULT_DIR / "plots"

RESULT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILE = BASE_DIR / "tvpvar_input_scaled.csv"

DATE_COL = "Date"
TARGET = "dlog_SOLVPN"

OTHERS = [
    "dlog_COPPER",
    "dlog_GOLD",
    "dlog_SILVER",
    "dlog_DXY",
    "dlog_OIL",
    "dlog_GAS",
    "d_UST10Y",
    "d_VIX",
    "d_PJM_LMP",
]

# Robustness check
MIN_SIZE_LIST = [30, 60, 90, 120]

# PELT penalty candidates
PENALTY_LIST = [5, 7, 10, 15, 20, 30, 50]

# normal: Gaussian regime assumption
# rbf: robustness candidate
MODEL_LIST = ["normal", "rbf"]

# ±WINDOW_DAYS 이내 change point를 같은 structural date cluster로 묶음
WINDOW_DAYS = 10

EPS = 1e-8

# -----------------------------
# Output files
# -----------------------------
OUT_GRID = RESULT_DIR / "pair_ggs_grid_search_bic_by_minsize.csv"
OUT_SELECTED = RESULT_DIR / "pair_ggs_selected_by_minsize.csv"
OUT_CP = RESULT_DIR / "pair_ggs_change_points_by_minsize.csv"
OUT_SEGMENTS = RESULT_DIR / "pair_ggs_segments_by_minsize.csv"
OUT_STATS = RESULT_DIR / "pair_ggs_segment_stats_by_minsize.csv"
OUT_ROBUST = RESULT_DIR / "pair_ggs_robust_change_point_clusters.csv"
OUT_PAIR_SUMMARY = RESULT_DIR / "pair_ggs_pair_summary.csv"


# ============================================================
# Data
# ============================================================
def load_data():
    df = pd.read_csv(INPUT_FILE)

    if DATE_COL not in df.columns:
        raise ValueError(f"Missing date column: {DATE_COL}")

    required_cols = [TARGET] + OTHERS
    missing = [c for c in required_cols if c not in df.columns]

    if missing:
        raise ValueError(f"Missing columns: {missing}")

    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")

    df = (
        df[[DATE_COL] + required_cols]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .sort_values(DATE_COL)
        .reset_index(drop=True)
    )

    if df.empty:
        raise ValueError("No usable data after cleaning.")

    return df


# ============================================================
# Segmentation
# ============================================================
def run_pelt(X, model_name, penalty, min_size):
    try:
        algo = rpt.Pelt(
            model=model_name,
            min_size=min_size,
        ).fit(X)

        cps = algo.predict(pen=penalty)

        # ruptures returns len(X) as final boundary
        change_idx = [int(cp) for cp in cps if cp < len(X)]

        return change_idx, ""

    except Exception as e:
        return [], repr(e)


def get_bounds(change_idx, n_obs):
    return [0] + list(change_idx) + [n_obs]


def get_segment_lengths(change_idx, n_obs):
    bounds = get_bounds(change_idx, n_obs)

    return [
        bounds[i + 1] - bounds[i]
        for i in range(len(bounds) - 1)
    ]


# ============================================================
# Gaussian NLL and BIC-style criterion
# ============================================================
def gaussian_segment_nll(X_seg):
    """
    Multivariate Gaussian negative log-likelihood up to constants.

    NLL = n/2 * log|Sigma|
        + 1/2 * sum Mahalanobis distance
    """
    X_seg = np.asarray(X_seg, dtype=float)

    n, d = X_seg.shape

    if n <= d + 1:
        return np.inf

    mu = X_seg.mean(axis=0)
    Xc = X_seg - mu

    Sigma = (Xc.T @ Xc) / n
    Sigma = Sigma + np.eye(d) * EPS

    sign, logdet = np.linalg.slogdet(Sigma)

    if sign <= 0 or not np.isfinite(logdet):
        return np.inf

    inv_Sigma = np.linalg.inv(Sigma)

    mahal = np.einsum(
        "ij,jk,ik->i",
        Xc,
        inv_Sigma,
        Xc,
    ).sum()

    nll = 0.5 * n * logdet + 0.5 * mahal

    return float(nll)


def total_gaussian_nll(X, change_idx):
    bounds = get_bounds(change_idx, len(X))
    total = 0.0

    for i in range(len(bounds) - 1):
        start = bounds[i]
        end = bounds[i + 1]

        nll = gaussian_segment_nll(X[start:end])

        if not np.isfinite(nll):
            return np.inf

        total += nll

    return float(total)


def bic_style_score(X, change_idx):
    """
    BIC-style score:
    score = 2 * NLL + q * log(n)

    q = total Gaussian parameters across all segments
      = K * [d mean parameters + d(d+1)/2 covariance parameters]

    K = number of segments = number of change points + 1
    """
    n, d = X.shape
    K = len(change_idx) + 1

    nll = total_gaussian_nll(X, change_idx)

    n_params_per_segment = d + (d * (d + 1)) / 2
    q = K * n_params_per_segment

    score = 2.0 * nll + q * np.log(n)

    return float(score), float(nll), int(q)


def choose_best_candidate_for_pair_minsize(grid_df, pair_name, min_size):
    valid = grid_df[
        (grid_df["pair"] == pair_name) &
        (grid_df["min_size"] == min_size) &
        (grid_df["error_msg"] == "") &
        np.isfinite(grid_df["bic_score"])
    ].copy()

    if valid.empty:
        raise RuntimeError(
            f"No valid candidate for pair={pair_name}, min_size={min_size}"
        )

    valid = valid.sort_values(
        ["bic_score", "n_cp", "min_segment_length"],
        ascending=[True, True, False],
    ).reset_index(drop=True)

    return valid.iloc[0]


# ============================================================
# Tables
# ============================================================
def make_change_point_table(pair_name, min_size, model_name, penalty, change_idx, dates):
    return pd.DataFrame({
        "pair": pair_name,
        "min_size": min_size,
        "model": model_name,
        "penalty": penalty,
        "cp_id": np.arange(1, len(change_idx) + 1),
        "change_idx": change_idx,
        "change_date": dates.iloc[change_idx].values,
    })


def make_segment_table(pair_name, min_size, model_name, penalty, change_idx, dates):
    bounds = get_bounds(change_idx, len(dates))

    rows = []

    for seg_id in range(len(bounds) - 1):
        start_idx = bounds[seg_id]
        end_exclusive = bounds[seg_id + 1]
        end_idx = end_exclusive - 1

        rows.append({
            "pair": pair_name,
            "min_size": min_size,
            "model": model_name,
            "penalty": penalty,
            "segment_id": seg_id,
            "start_idx": start_idx,
            "end_idx": end_idx,
            "start_date": dates.iloc[start_idx],
            "end_date": dates.iloc[end_idx],
            "length": end_exclusive - start_idx,
        })

    return pd.DataFrame(rows)


def make_segment_stats(pair_name, min_size, model_name, penalty, pair_cols, pair_df, segments_df):
    rows = []

    for _, seg in segments_df.iterrows():
        seg_id = int(seg["segment_id"])
        start_idx = int(seg["start_idx"])
        end_idx = int(seg["end_idx"])

        seg_data = pair_df.iloc[start_idx:end_idx + 1]

        for feature in pair_cols:
            values = seg_data[feature].values

            rows.append({
                "pair": pair_name,
                "min_size": min_size,
                "model": model_name,
                "penalty": penalty,
                "segment_id": seg_id,
                "feature": feature,
                "mean": float(np.mean(values)),
                "std": float(np.std(values, ddof=0)),
                "var": float(np.var(values, ddof=0)),
                "min": float(np.min(values)),
                "max": float(np.max(values)),
                "median": float(np.median(values)),
            })

    return pd.DataFrame(rows)


# ============================================================
# Robust cluster
# ============================================================
def make_robust_cp_clusters(cp_df, window_days=10):
    """
    pair별로 min_size별 change point를 날짜 기준으로 clustering.
    window_days 이내 날짜는 같은 structural date cluster로 간주.
    """
    if cp_df.empty:
        return pd.DataFrame(columns=[
            "pair",
            "cluster_id",
            "cluster_start",
            "cluster_end",
            "representative_date",
            "n_occurrences",
            "n_unique_min_size",
            "min_sizes",
            "dates",
        ])

    temp = cp_df.copy()
    temp["change_date"] = pd.to_datetime(temp["change_date"])

    all_rows = []

    for pair_name, g in temp.groupby("pair"):
        g = g.sort_values("change_date").reset_index(drop=True)

        if g.empty:
            continue

        clusters = []
        current = [g.iloc[0]]

        for i in range(1, len(g)):
            prev_date = pd.to_datetime(current[-1]["change_date"])
            curr_date = pd.to_datetime(g.iloc[i]["change_date"])

            if (curr_date - prev_date).days <= window_days:
                current.append(g.iloc[i])
            else:
                clusters.append(current)
                current = [g.iloc[i]]

        clusters.append(current)

        for cid, cluster in enumerate(clusters, start=1):
            cdf = pd.DataFrame(cluster)
            cdf["change_date"] = pd.to_datetime(cdf["change_date"])

            rep_date = cdf["change_date"].median()

            all_rows.append({
                "pair": pair_name,
                "cluster_id": cid,
                "cluster_start": cdf["change_date"].min(),
                "cluster_end": cdf["change_date"].max(),
                "representative_date": rep_date,
                "n_occurrences": len(cdf),
                "n_unique_min_size": cdf["min_size"].nunique(),
                "min_sizes": "|".join(map(str, sorted(cdf["min_size"].unique()))),
                "dates": "|".join(cdf["change_date"].astype(str).tolist()),
            })

    return pd.DataFrame(all_rows)


def make_pair_summary(robust_df):
    if robust_df.empty:
        return pd.DataFrame()

    rows = []

    for pair_name, g in robust_df.groupby("pair"):
        max_robustness = int(g["n_unique_min_size"].max())
        n_clusters = int(len(g))
        n_fully_robust = int((g["n_unique_min_size"] == len(MIN_SIZE_LIST)).sum())

        top_dates = (
            g.sort_values(
                ["n_unique_min_size", "representative_date"],
                ascending=[False, True],
            )["representative_date"]
            .astype(str)
            .tolist()
        )

        rows.append({
            "pair": pair_name,
            "n_clusters": n_clusters,
            "max_robustness": max_robustness,
            "n_fully_robust_clusters": n_fully_robust,
            "top_cluster_dates": "|".join(top_dates[:5]),
        })

    return pd.DataFrame(rows)


# ============================================================
# Plot
# ============================================================
def plot_pair_robust(pair_name, pair_cols, pair_df, cp_df_pair, robust_df_pair):
    dates = pair_df[DATE_COL]

    plt.figure(figsize=(14, 6))

    for col in pair_cols:
        plt.plot(
            dates,
            pair_df[col],
            linewidth=1.0,
            alpha=0.85,
            label=col,
        )

    # 개별 min_size별 CP
    if not cp_df_pair.empty:
        for _, row in cp_df_pair.iterrows():
            plt.axvline(
                pd.to_datetime(row["change_date"]),
                linestyle="--",
                linewidth=0.8,
                alpha=0.35,
            )

    # robust cluster representative CP
    if not robust_df_pair.empty:
        for _, row in robust_df_pair.iterrows():
            plt.axvline(
                pd.to_datetime(row["representative_date"]),
                linestyle="-",
                linewidth=2.0,
                alpha=0.9,
            )

    plt.title(f"Pairwise GGS Robust Change Points - {pair_name}")
    plt.xlabel("Date")
    plt.ylabel("Standardized value")
    plt.legend()
    plt.tight_layout()

    out_path = PLOT_DIR / f"{pair_name}.png"
    plt.savefig(out_path, dpi=300)
    plt.close()


# ============================================================
# Main
# ============================================================
def main():
    print("================================================")
    print("Pairwise GGS-like Robust Change Point Detection")
    print("min_size robustness: 30, 60, 90, 120")
    print("================================================")

    data = load_data()

    all_grid_rows = []
    selected_rows = []
    cp_tables = []
    segment_tables = []
    stat_tables = []

    # --------------------------------------------------------
    # Full grid search pair by pair
    # --------------------------------------------------------
    for other in OTHERS:
        pair_cols = [TARGET, other]
        pair_name = f"{TARGET}__{other}"

        print()
        print("PAIR:", pair_name)

        pair_df = data[[DATE_COL] + pair_cols].copy()
        X = pair_df[pair_cols].values
        dates = pair_df[DATE_COL]

        # -----------------------------
        # Grid search for this pair
        # -----------------------------
        pair_grid_rows = []

        for model_name in MODEL_LIST:
            for min_size in MIN_SIZE_LIST:
                for penalty in PENALTY_LIST:
                    change_idx, error_msg = run_pelt(
                        X=X,
                        model_name=model_name,
                        penalty=penalty,
                        min_size=min_size,
                    )

                    seg_lengths = get_segment_lengths(change_idx, len(X))

                    bic_score, nll, n_params = bic_style_score(
                        X,
                        change_idx,
                    )

                    pair_grid_rows.append({
                        "pair": pair_name,
                        "model": model_name,
                        "min_size": min_size,
                        "penalty": penalty,
                        "n_cp": len(change_idx),
                        "n_segments": len(change_idx) + 1,
                        "min_segment_length": min(seg_lengths),
                        "max_segment_length": max(seg_lengths),
                        "mean_segment_length": float(np.mean(seg_lengths)),
                        "gaussian_nll": nll,
                        "n_params": n_params,
                        "bic_score": bic_score,
                        "change_idx": "|".join(map(str, change_idx)),
                        "change_dates": "|".join(
                            dates.iloc[change_idx].astype(str).tolist()
                        ) if len(change_idx) > 0 else "",
                        "error_msg": error_msg,
                    })

        pair_grid_df = pd.DataFrame(pair_grid_rows)
        all_grid_rows.append(pair_grid_df)

        # -----------------------------
        # Select best for each min_size
        # -----------------------------
        for min_size in MIN_SIZE_LIST:
            best = choose_best_candidate_for_pair_minsize(
                pair_grid_df,
                pair_name=pair_name,
                min_size=min_size,
            )

            best_model = best["model"]
            best_penalty = float(best["penalty"])

            change_idx, error_msg = run_pelt(
                X=X,
                model_name=best_model,
                penalty=best_penalty,
                min_size=min_size,
            )

            if error_msg:
                raise RuntimeError(
                    f"Best candidate failed for pair={pair_name}, min_size={min_size}: {error_msg}"
                )

            seg_lengths = get_segment_lengths(change_idx, len(X))
            bic_score, nll, n_params = bic_style_score(X, change_idx)

            selected_rows.append({
                "pair": pair_name,
                "min_size": min_size,
                "selected_model": best_model,
                "selected_penalty": best_penalty,
                "n_cp": len(change_idx),
                "n_segments": len(change_idx) + 1,
                "min_segment_length": min(seg_lengths),
                "max_segment_length": max(seg_lengths),
                "mean_segment_length": float(np.mean(seg_lengths)),
                "gaussian_nll": nll,
                "n_params": n_params,
                "bic_score": bic_score,
                "change_idx": "|".join(map(str, change_idx)),
                "change_dates": "|".join(
                    dates.iloc[change_idx].astype(str).tolist()
                ) if len(change_idx) > 0 else "",
            })

            cp_tables.append(
                make_change_point_table(
                    pair_name=pair_name,
                    min_size=min_size,
                    model_name=best_model,
                    penalty=best_penalty,
                    change_idx=change_idx,
                    dates=dates,
                )
            )

            seg_df = make_segment_table(
                pair_name=pair_name,
                min_size=min_size,
                model_name=best_model,
                penalty=best_penalty,
                change_idx=change_idx,
                dates=dates,
            )

            segment_tables.append(seg_df)

            stat_tables.append(
                make_segment_stats(
                    pair_name=pair_name,
                    min_size=min_size,
                    model_name=best_model,
                    penalty=best_penalty,
                    pair_cols=pair_cols,
                    pair_df=pair_df,
                    segments_df=seg_df,
                )
            )

            print(
                f"  min_size={min_size:>3} | "
                f"model={best_model:<6} | "
                f"pen={best_penalty:<4} | "
                f"n_cp={len(change_idx):<2} | "
                f"segments={seg_lengths}"
            )

    # --------------------------------------------------------
    # Save grid and selected tables
    # --------------------------------------------------------
    final_grid = pd.concat(all_grid_rows, ignore_index=True)
    selected_df = pd.DataFrame(selected_rows)

    cp_df = pd.concat(cp_tables, ignore_index=True) if cp_tables else pd.DataFrame()
    segments_df = pd.concat(segment_tables, ignore_index=True) if segment_tables else pd.DataFrame()
    stats_df = pd.concat(stat_tables, ignore_index=True) if stat_tables else pd.DataFrame()

    # --------------------------------------------------------
    # Robust clusters
    # --------------------------------------------------------
    robust_df = make_robust_cp_clusters(
        cp_df,
        window_days=WINDOW_DAYS,
    )

    pair_summary_df = make_pair_summary(robust_df)

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------
    final_grid.to_csv(OUT_GRID, index=False, encoding="utf-8-sig")
    selected_df.to_csv(OUT_SELECTED, index=False, encoding="utf-8-sig")
    cp_df.to_csv(OUT_CP, index=False, encoding="utf-8-sig")
    segments_df.to_csv(OUT_SEGMENTS, index=False, encoding="utf-8-sig")
    stats_df.to_csv(OUT_STATS, index=False, encoding="utf-8-sig")
    robust_df.to_csv(OUT_ROBUST, index=False, encoding="utf-8-sig")
    pair_summary_df.to_csv(OUT_PAIR_SUMMARY, index=False, encoding="utf-8-sig")

    # --------------------------------------------------------
    # Plot pair-wise robust figures
    # --------------------------------------------------------
    for other in OTHERS:
        pair_cols = [TARGET, other]
        pair_name = f"{TARGET}__{other}"

        pair_df = data[[DATE_COL] + pair_cols].copy()

        cp_df_pair = cp_df[cp_df["pair"] == pair_name].copy()
        robust_df_pair = robust_df[robust_df["pair"] == pair_name].copy()

        plot_pair_robust(
            pair_name=pair_name,
            pair_cols=pair_cols,
            pair_df=pair_df,
            cp_df_pair=cp_df_pair,
            robust_df_pair=robust_df_pair,
        )

    # --------------------------------------------------------
    # Print summary
    # --------------------------------------------------------
    print()
    print("================================================")
    print("Saved Files")
    print("================================================")
    print(" -", OUT_GRID)
    print(" -", OUT_SELECTED)
    print(" -", OUT_CP)
    print(" -", OUT_SEGMENTS)
    print(" -", OUT_STATS)
    print(" -", OUT_ROBUST)
    print(" -", OUT_PAIR_SUMMARY)
    print(" -", PLOT_DIR)

    print()
    print("[Pair Summary]")
    if pair_summary_df.empty:
        print("No robust pair clusters.")
    else:
        print(pair_summary_df.to_string(index=False))

    print()
    print("Finished.")


if __name__ == "__main__":
    main()

Pairwise GGS-like Robust Change Point Detection
min_size robustness: 30, 60, 90, 120

PAIR: dlog_SOLVPN__dlog_COPPER
  min_size= 30 | model=normal | pen=30.0 | n_cp=2  | segments=[505, 30, 71]
  min_size= 60 | model=normal | pen=20.0 | n_cp=3  | segments=[410, 60, 60, 76]
  min_size= 90 | model=normal | pen=30.0 | n_cp=1  | segments=[505, 101]
  min_size=120 | model=normal | pen=20.0 | n_cp=1  | segments=[455, 151]

PAIR: dlog_SOLVPN__dlog_GOLD
  min_size= 30 | model=normal | pen=20.0 | n_cp=1  | segments=[455, 151]
  min_size= 60 | model=normal | pen=20.0 | n_cp=1  | segments=[455, 151]
  min_size= 90 | model=normal | pen=15.0 | n_cp=1  | segments=[455, 151]
  min_size=120 | model=normal | pen=10.0 | n_cp=1  | segments=[455, 151]

PAIR: dlog_SOLVPN__dlog_SILVER
  min_size= 30 | model=normal | pen=30.0 | n_cp=1  | segments=[555, 51]
  min_size= 60 | model=normal | pen=30.0 | n_cp=1  | segments=[545, 61]
  min_size= 90 | model=normal | pen=30.0 | n_cp=0  | segments=[606]
  min_size=120 